In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation

import jax.numpy as jnp
from jax.scipy.spatial.transform import Rotation as jaxRotation

import matplotlib.pyplot as plt

from helicopter.vision.tracking import TrianglePointMatcher, ICP

In [ ]:
measured_points = np.array([[      1.817, -0.00031891,    -0.43681],
       [     1.8537,    -0.14072,    -0.41775],
       [     1.8144,   -0.093097,    -0.42714],
       [     1.8105,    -0.11144,    -0.42388],
       [     1.8304,    -0.12465,    -0.41583],
       [     1.8697,    -0.16113,    -0.42603],
       [     1.8149,   -0.097907,    -0.41591],
       [     1.8176,   0.0084803,     -0.4476],
       [     1.8226,    -0.10631,    -0.45104],
       [     1.8169,    -0.10612,     -0.4095],
       [     1.8329,    -0.14354,    -0.42963]])

In [ ]:
reference_points = np.load('/home/ray/projects/helicopter/assets/point_clouds/green_syma.npy')

In [ ]:
def plot_3d(points, window, marker_size=15, c=None, show_indices=True, other_points=None):
    plt.close('all')

    fig = plt.figure(figsize=(8, 8))
    ax = fig.add_subplot(projection='3d')

    x = points[:, 0]
    y = points[:, 1]
    z = points[:, 2]

    if c is None:
        c = y

    ax.scatter(x, y, z, c=c, cmap='plasma', s=marker_size, depthshade=True)

    if show_indices:
        for i in range(len(points)):
            ax.text(x[i], y[i], z[i], str(i), size=8, color='black', zorder=10)

    if other_points is not None:
        x = other_points[:, 0]
        y = other_points[:, 1]
        z = other_points[:, 2]

        ax.scatter(x, y, z, c='g', s=marker_size, depthshade=True)

    ax.set_xlim(-window, window)
    ax.set_ylim(-window, window)
    ax.set_zlim(-window, window)

    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

    plt.show()

In [ ]:
matcher = TrianglePointMatcher(n=500, k=50)
r, t = matcher.get_alignment(sample_points=measured_points)

In [ ]:
r.as_rotvec(degrees=True)

In [ ]:
t

In [ ]:
%matplotlib ipympl

plot_3d(r.apply(reference_points), other_points=(measured_points - t), window=0.1)

In [ ]:
icp = ICP(distance_threshold=1e-1, etol=0.003, max_iterations=10)

In [ ]:
q_jax = jaxRotation.from_quat(jnp.array(r.as_quat(canonical=True)))
_, _, q_new, t_new = icp.iterate(q_old=q_jax, t_old=t, sample_points=measured_points, reference_points=reference_points)

In [ ]:
q_new.as_rotvec(degrees=True)

In [ ]:
%matplotlib ipympl
q_np = Rotation.from_quat(np.array(q_jax.as_quat(canonical=True)))
plot_3d(q_new.apply(reference_points), other_points=(measured_points - t), window=0.1)